In [0]:
import os
import requests
import json

from datetime import datetime
from zoneinfo import ZoneInfo

# ------------------------------------------------------------
# OPENROUTER API KEY
# ------------------------------------------------------------

os.environ["OPENROUTER_API_KEY"] = "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"

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is not configured.")

print("OpenRouter API key loaded successfully.")




# ============================================================
# COMPLETE WEATHER + OPENROUTER PIPELINE
# DATABRICKS - SINGLE CELL
# WITHOUT SECRET SCOPE
# ============================================================

import os
import requests
import json

from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)


# ============================================================
# 1. GET TODAY'S DATE - INDIA TIMEZONE
# ============================================================

india_timezone = ZoneInfo("Asia/Kolkata")

today_date = (
    datetime
    .now(india_timezone)
    .date()
    .isoformat()
)

print("Fetching weather data for:", today_date)


# ============================================================
# 2. OPEN-METEO API SETTINGS
# ============================================================

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"


weather_params = {

    "latitude": 21.034408,
    "longitude": 79.032069,

    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "apparent_temperature,"
        "precipitation_probability,"
        "precipitation,"
        "rain,"
        "showers,"
        "weather_code,"
        "cloud_cover,"
        "visibility,"
        "pressure_msl,"
        "surface_pressure,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "wind_gusts_10m"
    ),

    "timezone": "Asia/Kolkata",

    "start_date": today_date,
    "end_date": today_date
}


# ============================================================
# 3. CALL OPEN-METEO API
# ============================================================

try:

    weather_response = requests.get(
        WEATHER_URL,
        params=weather_params,
        timeout=30
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()["hourly"]

except requests.exceptions.RequestException as error:

    raise RuntimeError(
        f"Open-Meteo API failed: {error}"
    )


# ============================================================
# 4. CONVERT WEATHER DATA INTO ROWS
# ============================================================

hourly_rows = list(
    zip(
        weather_data["time"],
        weather_data["temperature_2m"],
        weather_data["relative_humidity_2m"],
        weather_data["dew_point_2m"],
        weather_data["apparent_temperature"],
        weather_data["precipitation_probability"],
        weather_data["precipitation"],
        weather_data["rain"],
        weather_data["showers"],
        weather_data["weather_code"],
        weather_data["cloud_cover"],
        weather_data["visibility"],
        weather_data["pressure_msl"],
        weather_data["surface_pressure"],
        weather_data["wind_speed_10m"],
        weather_data["wind_direction_10m"],
        weather_data["wind_gusts_10m"]
    )
)


# ============================================================
# 5. WEATHER SCHEMA
# ============================================================

weather_schema = """
time STRING,
temperature_2m DOUBLE,
relative_humidity_2m LONG,
dew_point_2m DOUBLE,
apparent_temperature DOUBLE,
precipitation_probability LONG,
precipitation DOUBLE,
rain DOUBLE,
showers DOUBLE,
weather_code LONG,
cloud_cover LONG,
visibility DOUBLE,
pressure_msl DOUBLE,
surface_pressure DOUBLE,
wind_speed_10m DOUBLE,
wind_direction_10m LONG,
wind_gusts_10m DOUBLE
"""


# ============================================================
# 6. CREATE WEATHER SPARK DATAFRAME
# ============================================================

df_today = spark.createDataFrame(
    hourly_rows,
    schema=weather_schema
)


weather_row_count = df_today.count()

print("Total weather rows:", weather_row_count)


if weather_row_count == 0:
    raise ValueError(
        "No weather data was returned."
    )


# display(df_today)


# ============================================================
# 7. CONVERT WEATHER DATA TO JSON
# ============================================================

weather_records = [
    row.asDict(recursive=True)
    for row in df_today.collect()
]


today_weather_json = json.dumps(
    weather_records,
    indent=2,
    default=str
)


# ============================================================
# 8. GET OPENROUTER API KEY FROM ENVIRONMENT VARIABLE
# ============================================================

api_key = api_key


if not api_key:

    raise ValueError(
        """
OPENROUTER_API_KEY is not configured.

Create an environment variable named:

OPENROUTER_API_KEY

and store your OpenRouter API key inside it.
"""
    )


print("OpenRouter API key loaded successfully.")


# ============================================================
# 9. OPENROUTER API URL
# ============================================================

OPENROUTER_URL = (
    "https://openrouter.ai/api/v1/chat/completions"
)


# ============================================================
# 10. OPENROUTER HEADERS
# ============================================================

headers = {

    "Authorization": f"Bearer {api_key}",

    "Content-Type": "application/json"

}


# ============================================================
# 11. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are a weather recommendation assistant.

Read today's hourly weather forecast carefully.

Generate exactly 3 short and practical recommendations
based only on the provided weather data.

Consider:

- temperature
- apparent temperature
- humidity
- precipitation probability
- precipitation
- rain
- showers
- cloud cover
- wind speed
- wind gusts
- visibility

Rules:

1. Return ONLY a valid JSON array.
2. The JSON array must contain exactly 3 objects.
3. Every object must contain exactly two fields:
   "sr_no" and "recommendation".
4. The sr_no values must be exactly 1, 2 and 3.
5. Keep every recommendation short and practical.
6. Base recommendations only on supplied weather data.
7. Do not invent weather conditions.
8. Do not return Markdown.
9. Do not return code fences.
10. Do not return headings.
11. Do not return explanations.
12. Do not return summaries.
13. Do not return text outside the JSON array.

Required output:

[
  {
    "sr_no": 1,
    "recommendation": "Recommendation text"
  },
  {
    "sr_no": 2,
    "recommendation": "Recommendation text"
  },
  {
    "sr_no": 3,
    "recommendation": "Recommendation text"
  }
]
"""


# ============================================================
# 12. USER MESSAGE
# ============================================================

USER_MESSAGE = f"""
Today's hourly weather forecast data is provided below.

WEATHER DATA:

{today_weather_json}

Analyze the weather data above and return exactly
3 practical recommendations.

Follow the JSON format specified in the system instructions.
"""


# ============================================================
# 13. CREATE OPENROUTER MESSAGES
# ============================================================

messages = [

    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },

    {
        "role": "user",
        "content": USER_MESSAGE
    }

]


# ============================================================
# 14. CREATE OPENROUTER PAYLOAD
# ============================================================

payload = {

    "model": "openrouter/free",

    "messages": messages

}


# ============================================================
# 15. CALL OPENROUTER API
# ============================================================

try:

    openrouter_response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload,
        timeout=60
    )


    # print(
    #     "OpenRouter Status Code:",
    #     openrouter_response.status_code
    # )


    openrouter_response.raise_for_status()


    response_data = openrouter_response.json()


except requests.exceptions.Timeout:

    raise RuntimeError(
        "OpenRouter request timed out."
    )


except requests.exceptions.HTTPError:

    raise RuntimeError(
        f"""
OpenRouter API Error

Status Code:
{openrouter_response.status_code}

Response:
{openrouter_response.text}
"""
    )


except requests.exceptions.RequestException as error:

    raise RuntimeError(
        f"OpenRouter connection error: {error}"
    )


# ============================================================
# 16. EXTRACT AI RESPONSE
# ============================================================

try:

    ai_output = (
        response_data["choices"][0]
        ["message"]["content"]
    )

except (
    KeyError,
    IndexError,
    TypeError
):

    raise RuntimeError(
        f"""
Unexpected OpenRouter response:

{response_data}
"""
    )


# print("\nRaw AI Response:")
# print(ai_output)


# ============================================================
# 17. CLEAN AI RESPONSE
# ============================================================

clean_ai_output = (
    ai_output
    .strip()
    .replace("```json", "")
    .replace("```JSON", "")
    .replace("```", "")
    .strip()
)


# ============================================================
# 18. CONVERT AI RESPONSE TO PYTHON JSON
# ============================================================

try:

    recommendations = json.loads(
        clean_ai_output
    )

except json.JSONDecodeError as error:

    raise ValueError(
        f"""
AI did not return valid JSON.

AI Response:

{ai_output}

JSON Error:

{error}
"""
    )


# ============================================================
# 19. VALIDATE JSON ARRAY
# ============================================================

if not isinstance(
    recommendations,
    list
):

    raise ValueError(
        "AI response must be a JSON array."
    )


# ============================================================
# 20. VALIDATE EXACTLY 3 RECOMMENDATIONS
# ============================================================

if len(recommendations) != 3:

    raise ValueError(
        f"""
Expected exactly 3 recommendations.

Received:
{len(recommendations)}
"""
    )


# ============================================================
# 21. VALIDATE REQUIRED FIELDS
# ============================================================

expected_fields = {
    "sr_no",
    "recommendation"
}


for item in recommendations:

    if not isinstance(item, dict):

        raise ValueError(
            "Every recommendation must be a JSON object."
        )


    if set(item.keys()) != expected_fields:

        raise ValueError(
            f"""
Invalid fields received.

Expected:
{expected_fields}

Received:
{set(item.keys())}
"""
        )


# ============================================================
# 22. VALIDATE SERIAL NUMBERS
# ============================================================

expected_sr_numbers = [
    1,
    2,
    3
]


actual_sr_numbers = [
    item["sr_no"]
    for item in recommendations
]


if actual_sr_numbers != expected_sr_numbers:

    raise ValueError(
        f"""
Invalid sr_no values.

Expected:
{expected_sr_numbers}

Received:
{actual_sr_numbers}
"""
    )


# ============================================================
# 23. RECOMMENDATION SPARK SCHEMA
# ============================================================

recommendation_schema = StructType([

    StructField(
        "sr_no",
        IntegerType(),
        False
    ),

    StructField(
        "recommendation",
        StringType(),
        False
    )

])


# ============================================================
# 24. CREATE RECOMMENDATION SPARK DATAFRAME
# ============================================================

recommendation_df = spark.createDataFrame(
    recommendations,
    schema=recommendation_schema
)


# print("\nFinal AI Recommendations:")

# display(recommendation_df)


# ============================================================
# 25. SAVE WEATHER DELTA TABLE
# ============================================================

df_today.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.weather_today"
    )


# ============================================================
# 26. SAVE AI RECOMMENDATIONS DELTA TABLE
# ============================================================

recommendation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.weather_bronze.weather_recommendations"
    )


# ============================================================
# 27. FINAL STATUS
# ============================================================

# print("\n------------------------------------------")
# print("Pipeline completed successfully.")
# print("------------------------------------------")

# print(
#     "Weather Delta Table:",
#     "workspace.default.weather_today"
# )

# print(
#     "Recommendation Delta Table:",
#     "workspace.default.weather_recommendations"
# )

In [0]:
display(recommendation_df)